In [ ]:
import numpy as np
import xarray as xr
from datetime import datetime, timedelta
# from scipy import stats

In [ ]:
import pickle

In [ ]:
from dask.distributed import Client

In [ ]:
# somehow without explicitly specifying 1 thread per worker, I get a lot of non-fatal errors out of xarray or netcdf4.
client = Client(n_workers=32, threads_per_worker=1)

In [ ]:
client

In [ ]:
times = np.arange(48000, 72001, 100)
files = [f"OUT_3D/BOMEX_r1_256_{t:010d}.nc" for t in times]
sam = xr.open_mfdataset(files, parallel=True, data_vars='all')
# minutes = np.rint((sam.time - 173.0) * 24.0 * 60.0)
qt = sam.QV + sam.QN
wi = sam.W
# w = np.zeros_like(wi)
w = (wi.shift(z=-1, fill_value = 0.0)+ wi) * 0.5
qcl = sam.QN
tr = sam.TR01
qtm = qt.mean(axis=(0, 2, 3))
qts = qt.std(axis=(0, 2, 3))
wm = w.mean(axis=(0, 2, 3))
ws = w.std(axis=(0, 2, 3))
trm = tr.mean(axis=(0, 2, 3))
trs = tr.std(axis=(0, 2, 3))
tv = sam.TABS * (1.0 + 0.608 * sam.QV * 0.001 - sam.QN * 0.001)
tvm = tv.mean(axis=(0, 2, 3))
qclm = qcl.mean(axis=(0, 2, 3))

In [ ]:
with open('pkl/wm.pkl', 'wb') as f:
    pickle.dump(wm.compute(), f)
with open('pkl/ws.pkl', 'wb') as f:
    pickle.dump(ws.compute(), f)
with open('pkl/trm.pkl', 'wb') as f:
    pickle.dump(trm.compute(), f)
with open('pkl/trs.pkl', 'wb') as f:
    pickle.dump(trs.compute(), f)
with open('pkl/qtm.pkl', 'wb') as f:
    pickle.dump(qtm.compute(), f)
with open('pkl/qts.pkl', 'wb') as f:
    pickle.dump(qts.compute(), f)
with open('pkl/tvm.pkl', 'wb') as f:
    pickle.dump(tvm.compute(), f)
with open('pkl/qclm.pkl', 'wb') as f:
    pickle.dump(qclm.compute(), f)